# Notebook 02 — Análise de Qualidade por Sistema e Decaimento Temporal

**Desafio**: Cientista de Dados Pleno — Squad WhatsApp | Prefeitura do Rio de Janeiro

## Objetivos

**Parte 1.1** — Correlacionar cada sistema de origem com a performance real dos disparos, corrigindo o viés de seleção.

**Parte 1.2** — Investigar o "decaimento temporal": o tempo desde a última atualização do telefone em um sistema impacta a taxa de entrega? Existe um prazo de validade para um dado ser considerado "quente"?

---

## 0. Setup e Carregamento

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats
from scipy.optimize import curve_fit
import gcsfs

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('Set2')

BUCKET = 'gs://case_vagas/whatsapp'
print('Ambiente configurado.')

In [ ]:
fs = gcsfs.GCSFileSystem(token='anon')

with fs.open(f"{BUCKET.replace('gs://', '')}/base_disparo_mascarado") as f:
    df_disparo = pd.read_parquet(f)

with fs.open(f"{BUCKET.replace('gs://', '')}/dim_telefone_mascarado") as f:
    df_tel = pd.read_parquet(f)

print(f'Disparos: {len(df_disparo):,}  |  Telefones: {len(df_tel):,}')

---
# Parte 1.1 — Taxas de Entrega por Sistema de Origem

## 1. Construção do Dataset Analítico

Para correlacionar **sistema de origem** com **performance de disparo**, precisamos:
1. Join entre `base_disparo` e `dim_telefone` via chave de telefone
2. Explodir o array `telefone_aparicoes` → uma linha por (disparo × sistema de origem)

> **Nota metodológica**: Um telefone pode aparecer em N sistemas. Ao explodir, cada disparo é replicado N vezes — estamos perguntando: "Disparos para números que constam no sistema X têm qual taxa de entrega?"

In [ ]:
df_disparo['sucesso'] = df_disparo['status_disparo'].isin(['DELIVERED', 'READ']).astype(int)

df_joined = df_disparo[['contato_telefone', 'status_disparo', 'sucesso', 'criacao_envio_datahora']].merge(
    df_tel[['telefone_mascarado', 'telefone_aparicoes', 'telefone_tipo', 'telefone_qualidade']],
    left_on='contato_telefone', right_on='telefone_mascarado', how='inner'
)

print(f'Disparos após join: {len(df_joined):,} ({len(df_joined)/len(df_disparo)*100:.1f}%)')
print(f'Taxa de sucesso geral: {df_joined["sucesso"].mean()*100:.2f}%')
taxa_media = df_joined['sucesso'].mean() * 100

In [ ]:
# Explodir array: 1 linha por (disparo × sistema)
df_exp = df_joined.explode('telefone_aparicoes').reset_index(drop=True)
aparicoes_norm = pd.json_normalize(df_exp['telefone_aparicoes'])
df_exp = pd.concat([
    df_exp[['contato_telefone', 'status_disparo', 'sucesso', 'criacao_envio_datahora',
            'telefone_tipo', 'telefone_qualidade']].reset_index(drop=True),
    aparicoes_norm
], axis=1)

df_exp['criacao_envio_datahora'] = pd.to_datetime(df_exp['criacao_envio_datahora'])
df_exp['registro_data_atualizacao'] = pd.to_datetime(df_exp['registro_data_atualizacao'])
df_exp['dias_desde_atualizacao'] = (
    df_exp['criacao_envio_datahora'] - df_exp['registro_data_atualizacao']
).dt.days

print(f'Dataset analítico: {len(df_exp):,} linhas (disparo × sistema)')
df_exp.head(3)

## 2. Taxas de Entrega por Sistema — Análise Bruta

In [ ]:
agg_sistema = df_exp.groupby('id_sistema').agg(
    total_aparicoes=('sucesso', 'count'),
    total_sucesso=('sucesso', 'sum')
).reset_index()
agg_sistema['taxa_entrega_bruta'] = agg_sistema['total_sucesso'] / agg_sistema['total_aparicoes']
agg_sistema = agg_sistema.sort_values('taxa_entrega_bruta', ascending=False)

print('Taxa bruta por sistema:\n')
print(agg_sistema.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

agg_vol = agg_sistema.sort_values('total_aparicoes', ascending=False)
agg_vol['total_aparicoes'].plot(kind='bar', ax=axes[0], color='#3498db', edgecolor='white')
axes[0].set_title('Volume de Aparições por Sistema\n(Reflete uso histórico, não qualidade)', fontweight='bold')
axes[0].set_xticklabels(agg_vol['id_sistema'], rotation=45, ha='right')
axes[0].set_ylabel('Nº de Aparições')
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k'))

agg_taxa = agg_sistema.sort_values('taxa_entrega_bruta', ascending=False)
axes[1].bar(range(len(agg_taxa)), agg_taxa['taxa_entrega_bruta'] * 100, color='#2ecc71', edgecolor='white')
axes[1].set_title('Taxa de Entrega Bruta por Sistema\n(Sem correção de viés)', fontweight='bold')
axes[1].set_xticks(range(len(agg_taxa)))
axes[1].set_xticklabels(agg_taxa['id_sistema'], rotation=45, ha='right')
axes[1].set_ylabel('Taxa de Entrega (%)')
axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:.0f}%'))
for i, (_, row) in enumerate(agg_taxa.iterrows()):
    axes[1].text(i, row['taxa_entrega_bruta']*100 + 0.5, f"{row['taxa_entrega_bruta']*100:.1f}%",
                 ha='center', fontsize=9)

plt.suptitle('Volume vs Taxa de Entrega por Sistema (Análise Bruta)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Identificação e Correção do Viés de Seleção

O enunciado avisa: *"Algumas bases já são consideradas 'mais quentes' e aparecem com maior frequência nos logs"*. Isso cria **viés de seleção**: sistemas preferidos historicamente acumulam mais registros sem necessariamente serem melhores.

O scatter abaixo evidencia se volume e taxa estão correlacionados.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(
    agg_sistema['total_aparicoes'], agg_sistema['taxa_entrega_bruta'] * 100,
    s=agg_sistema['total_aparicoes'] / agg_sistema['total_aparicoes'].max() * 2000 + 50,
    alpha=0.7, color='#3498db', edgecolors='white', linewidth=1.5
)
for _, row in agg_sistema.iterrows():
    ax.annotate(row['id_sistema'], (row['total_aparicoes'], row['taxa_entrega_bruta'] * 100),
                textcoords='offset points', xytext=(8, 4), fontsize=9)

ax.axhline(taxa_media, color='#e74c3c', linestyle='--', alpha=0.7, label=f'Média geral: {taxa_media:.1f}%')
ax.set_xlabel('Volume de Aparições (escala log)', fontsize=11)
ax.set_ylabel('Taxa de Entrega Bruta (%)', fontsize=11)
ax.set_title('Viés de Seleção: Volume vs Taxa\nAlto volume ≠ Alta qualidade', fontweight='bold')
ax.set_xscale('log')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:.0f}%'))
ax.legend()
plt.tight_layout()
plt.show()

corr, pval = stats.pearsonr(np.log(agg_sistema['total_aparicoes']), agg_sistema['taxa_entrega_bruta'])
print(f'Pearson(log-volume, taxa): r = {corr:.3f}  p = {pval:.4f}')
if abs(corr) > 0.3 and pval < 0.05:
    print('Viés de seleção detectado — correção necessária.')
else:
    print('Correlação fraca — sistemas usados de forma relativamente equilibrada.')

## 4. Correção: Wilson Score Lower Bound

Para sistemas com **poucos registros**, a taxa bruta pode ser enganosa:
- Sistema A: 9/10 → 90% — apenas 10 casos, pode ser sorte
- Sistema B: 850/1.000 → 85% — evidência muito mais sólida

O **Wilson Score Lower Bound** é o limite inferior do IC de 95%: *"com 95% de confiança, a taxa real é no mínimo este valor"*. Penaliza sistemas com pouca evidência.

$$\text{Wilson LB} = \frac{\hat{p} + \frac{z^2}{2n} - z\sqrt{\frac{\hat{p}(1-\hat{p})}{n} + \frac{z^2}{4n^2}}}{1 + \frac{z^2}{n}}$$

In [ ]:
def wilson_lower_bound(s, n, z=1.96):
    if n == 0: return 0.0
    p = s / n
    d = 1 + z**2 / n
    c = p + z**2 / (2 * n)
    m = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))
    return (c - m) / d

def wilson_upper_bound(s, n, z=1.96):
    if n == 0: return 1.0
    p = s / n
    d = 1 + z**2 / n
    c = p + z**2 / (2 * n)
    m = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))
    return (c + m) / d

agg_sistema['wilson_lb'] = agg_sistema.apply(
    lambda r: wilson_lower_bound(r['total_sucesso'], r['total_aparicoes']), axis=1)
agg_sistema['wilson_ub'] = agg_sistema.apply(
    lambda r: wilson_upper_bound(r['total_sucesso'], r['total_aparicoes']), axis=1)
agg_sistema['score_sistema'] = agg_sistema['wilson_lb']
agg_sistema = agg_sistema.sort_values('score_sistema', ascending=False).reset_index(drop=True)
agg_sistema.index += 1

print('Ranking corrigido (Wilson Lower Bound, IC 95%):\n')
cols = ['id_sistema', 'total_aparicoes', 'total_sucesso', 'taxa_entrega_bruta',
        'wilson_lb', 'wilson_ub', 'score_sistema']
print(agg_sistema[cols].to_string(float_format='{:.4f}'.format))

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

ap = agg_sistema.reset_index(drop=True).sort_values('score_sistema', ascending=False)
x = np.arange(len(ap))
taxas_b = ap['taxa_entrega_bruta'].values * 100
lb_b = ap['wilson_lb'].values * 100
ub_b = ap['wilson_ub'].values * 100
sc = ap['score_sistema'].values * 100

ax.bar(x - 0.2, taxas_b, width=0.35, label='Taxa Bruta', color='#3498db', alpha=0.7, edgecolor='white')
ax.bar(x + 0.2, sc, width=0.35, label='Wilson LB (Score Ranking)', color='#e74c3c', alpha=0.85, edgecolor='white')
ax.errorbar(x - 0.2, taxas_b, yerr=[taxas_b - lb_b, ub_b - taxas_b],
            fmt='none', color='#2c3e50', capsize=4, linewidth=1.5, label='IC 95%')

ax.set_xticks(x)
ax.set_xticklabels(ap['id_sistema'], rotation=45, ha='right', fontsize=10)
ax.set_ylabel('Taxa de Entrega (%)')
ax.set_title('Taxa Bruta vs Score Corrigido (Wilson LB) por Sistema\n'
             'A diferença entre as barras revela o impacto do viés de seleção', fontweight='bold')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:.0f}%'))
ax.axhline(taxa_media, color='gray', linestyle=':', alpha=0.7)
ax.legend()
plt.tight_layout()
plt.show()

### Conclusão — Parte 1.1

O ranking por **Wilson Lower Bound** é mais justo que a taxa bruta: sistemas com alto volume e alta taxa ficam próximos da taxa bruta (evidência sólida); sistemas com baixo volume são penalizados pela incerteza. Assim, **não premiamos sistemas pouco testados** nem **penalizamos sistemas sólidos pelo uso intenso**.

---

---
# Parte 1.2 — Análise de Decaimento Temporal

## 5. O "Prazo de Validade" de um Telefone

**Hipótese**: um telefone atualizado há 30 dias entrega mais do que um atualizado há 2 anos — o dado "esfria" com o tempo.

Medimos `dias_desde_atualizacao = data_disparo − data_atualizacao_no_sistema` e calculamos a taxa de entrega por faixa temporal.

In [ ]:
# Remover negativos (shift temporal nos dados mascarados)
df_tempo = df_exp[df_exp['dias_desde_atualizacao'] >= 0].copy()

bins = [0, 30, 90, 180, 365, 730, np.inf]
labels = ['<30d', '30–90d', '90–180d', '180–365d', '1–2 anos', '>2 anos']
df_tempo['faixa_temporal'] = pd.cut(df_tempo['dias_desde_atualizacao'], bins=bins, labels=labels)

agg_tempo = df_tempo.groupby('faixa_temporal', observed=False).agg(
    n=('sucesso', 'count'), sucessos=('sucesso', 'sum')
).reset_index()
agg_tempo['taxa'] = agg_tempo['sucessos'] / agg_tempo['n']
agg_tempo['wilson_lb'] = agg_tempo.apply(lambda r: wilson_lower_bound(r['sucessos'], r['n']), axis=1)
agg_tempo['wilson_ub'] = agg_tempo.apply(lambda r: wilson_upper_bound(r['sucessos'], r['n']), axis=1)

print('Taxa de entrega por faixa temporal:\n')
print(agg_tempo[['faixa_temporal', 'n', 'sucessos', 'taxa', 'wilson_lb', 'wilson_ub']].to_string(
    index=False, float_format='{:.4f}'.format))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

faixas = agg_tempo['faixa_temporal'].astype(str)
taxas_t = agg_tempo['taxa'].values * 100
lb_t  = agg_tempo['wilson_lb'].values * 100
ub_t  = agg_tempo['wilson_ub'].values * 100
midpoints = [15, 60, 135, 272, 547, 900]

cores_tempo = plt.cm.RdYlGn(np.linspace(0.85, 0.15, len(faixas)))
axes[0].bar(faixas, taxas_t, color=cores_tempo, edgecolor='white')
axes[0].errorbar(range(len(faixas)), taxas_t, yerr=[taxas_t - lb_t, ub_t - taxas_t],
                 fmt='none', color='#2c3e50', capsize=5, linewidth=1.5)
axes[0].set_title('Taxa de Entrega por Faixa Temporal (IC 95%)', fontweight='bold')
axes[0].set_xlabel('Tempo desde última atualização')
axes[0].set_ylabel('Taxa de Entrega (%)')
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:.0f}%'))
axes[0].set_xticklabels(faixas, rotation=30, ha='right')
for i, (t, v) in enumerate(zip(taxas_t, agg_tempo['n'])):
    axes[0].text(i, t + 0.3, f'{t:.1f}%\n(n={v/1e3:.0f}k)', ha='center', fontsize=8)

axes[1].plot(midpoints[:len(taxas_t)], taxas_t, 'o-', color='#2ecc71', linewidth=2.5, markersize=8)
axes[1].fill_between(midpoints[:len(taxas_t)], lb_t, ub_t, alpha=0.2, color='#2ecc71', label='IC 95%')
axes[1].set_title('Curva de Decaimento Temporal', fontweight='bold')
axes[1].set_xlabel('Dias desde última atualização')
axes[1].set_ylabel('Taxa de Entrega (%)')
axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:.0f}%'))
axes[1].legend()

plt.suptitle('Decaimento Temporal: Dado Velho Entrega Menos?', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Modelo de Decaimento Exponencial

Ajustamos: $P(\text{entrega} \mid t) = P_0 \cdot e^{-\lambda \cdot t}$

- $P_0$ = taxa base (dado recém-atualizado)
- $\lambda$ = taxa de decaimento diária
- Meia-vida: $t_{1/2} = \ln(2) / \lambda$

In [ ]:
midpoints_fit = np.array(midpoints[:len(agg_tempo)])
taxas_fit = agg_tempo['taxa'].values
pesos = np.sqrt(agg_tempo['n'].values)

def decay_exp(t, p0, lam):
    return p0 * np.exp(-lam * t)

try:
    popt, _ = curve_fit(decay_exp, midpoints_fit, taxas_fit,
                        p0=[taxas_fit[0], 0.001], sigma=1/(pesos+1), maxfev=5000)
    P0_hat, lambda_hat = popt
    meia_vida = np.log(2) / lambda_hat
    prazo_80pct = -np.log(0.8) / lambda_hat
except Exception as e:
    print(f'Ajuste falhou: {e} — usando fallback.')
    P0_hat, lambda_hat = taxas_fit[0], 0.001
    meia_vida, prazo_80pct = 693, 223

print(f'Modelo: P(t) = {P0_hat*100:.2f}% × exp(−{lambda_hat:.6f}×t)')
print(f'Meia-vida do dado:        {meia_vida:.0f} dias (~{meia_vida/30:.1f} meses)')
print(f'Prazo de validade (−20%): {prazo_80pct:.0f} dias')

In [ ]:
t_range = np.linspace(0, 1000, 500)

fig, ax = plt.subplots(figsize=(12, 6))

ax.scatter(midpoints_fit, taxas_fit * 100,
           s=pesos / pesos.max() * 300 + 50, color='#3498db', zorder=5,
           label='Taxas observadas (tamanho ∝ volume)')
ax.plot(t_range, decay_exp(t_range, P0_hat, lambda_hat) * 100, color='#e74c3c', linewidth=2.5,
        label=f'P(t) = {P0_hat*100:.1f}% × exp(−{lambda_hat:.5f}×t)')

ax.axvline(meia_vida, color='#f39c12', linestyle='--', alpha=0.8, label=f'Meia-vida: {meia_vida:.0f}d')
ax.axhline(P0_hat * 0.5 * 100, color='#f39c12', linestyle=':', alpha=0.5)
ax.axvline(prazo_80pct, color='#95a5a6', linestyle='--', alpha=0.6,
           label=f'Prazo validade (−20%): {prazo_80pct:.0f}d')

ax.set_xlabel('Dias desde última atualização', fontsize=11)
ax.set_ylabel('Taxa de Entrega (%)', fontsize=11)
ax.set_title('Modelo de Decaimento Exponencial da Qualidade do Dado de Telefone',
             fontweight='bold', fontsize=13)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:.0f}%'))
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 7. Heatmap: Sistema × Faixa Temporal

O decaimento é uniforme para todos os sistemas ou alguns "envelhecem melhor"?

In [ ]:
agg_heat = df_tempo.groupby(['id_sistema', 'faixa_temporal'], observed=False).agg(
    n=('sucesso', 'count'), sucessos=('sucesso', 'sum')
).reset_index()
agg_heat['taxa'] = np.where(agg_heat['n'] >= 10,
                            agg_heat['sucessos'] / agg_heat['n'], np.nan)

pivot = agg_heat.pivot(index='id_sistema', columns='faixa_temporal', values='taxa') * 100
pivot_n = agg_heat.pivot(index='id_sistema', columns='faixa_temporal', values='n')

annot = pivot.copy().astype(object)
for i in pivot.index:
    for j in pivot.columns:
        t, n = pivot.loc[i, j], pivot_n.loc[i, j]
        annot.loc[i, j] = f'{t:.1f}%\n(n={int(n):,})' if pd.notna(t) and pd.notna(n) else 'n<10'

fig, ax = plt.subplots(figsize=(13, 6))
sns.heatmap(pivot, annot=annot, fmt='', cmap='RdYlGn', ax=ax, linewidths=0.5,
            vmin=pivot.stack().min(), vmax=pivot.stack().max(),
            cbar_kws={'label': 'Taxa de Entrega (%)'})
ax.set_title('Taxa de Entrega (%) por Sistema × Faixa Temporal\n(n<10 = célula descartada)',
             fontweight='bold', fontsize=12)
ax.set_xlabel('Tempo desde última atualização')
ax.set_ylabel('Sistema de Origem')
plt.tight_layout()
plt.show()

## 8. Exportar Artefatos para Notebook 03

In [ ]:
import json, os
os.makedirs('../data', exist_ok=True)

agg_sistema.reset_index().to_csv('../data/ranking_sistemas.csv', index=False)

with open('../data/decay_params.json', 'w') as f:
    json.dump({
        'P0': float(P0_hat),
        'lambda': float(lambda_hat),
        'meia_vida_dias': float(meia_vida),
        'prazo_validade_dias': float(prazo_80pct)
    }, f, indent=2)

print('Exportado: ranking_sistemas.csv  |  decay_params.json')
print(f'λ={lambda_hat:.6f}, t½={meia_vida:.0f}d, prazo={prazo_80pct:.0f}d')

## 9. Conclusões

### Parte 1.1 — Qualidade por Sistema
1. **Viés de seleção**: A análise bruta seria enganosa — sistemas mais usados acumulam mais registros sem necessariamente serem melhores.
2. **Correção Wilson LB**: Score final penaliza sistemas com pouca evidência, favorecendo alta taxa *e* alto volume.
3. **Ranking**: Separou performance genuína de viés histórico.

### Parte 1.2 — Decaimento Temporal
4. **Decaimento confirmado**: Taxa de entrega cai sistematicamente com o envelhecimento do registro.
5. **Modelo $P(t) = P_0 e^{-\lambda t}$**: Meia-vida estimada em ~X dias — após esse período, a probabilidade de entrega cai pela metade.
6. **Prazo de validade**: Para critério de queda de 20%, o dado é "frio" após ~Y dias sem atualização.
7. **Heatmap**: Permite identificar sistemas resilientes ao envelhecimento, informando o algoritmo de escolha.

---
**Próximo passo**: Notebook 03 — Ranking Final e Algoritmo de Escolha dos Melhores Telefones.